# Memory refactoring

In long-running AI agents, memory stores accumulate entries over thousands of interactions. What starts as a lean, well-organized set of facts gradually becomes bloated with redundant variations of the same information, outdated observations, and overly specific entries that could be expressed as a single rule. Left unaddressed, this memory bloat degrades retrieval quality - the most relevant memories get crowded out by near-duplicates - increases storage overhead, and slows down context assembly at inference time.

Memory refactoring is a targeted maintenance strategy that periodically reorganizes stored memories without losing essential information. It is distinct from memory cleanup (which removes old or low-priority items) in that refactoring focuses on restructuring rather than deletion. The goal is to preserve the same semantic coverage with fewer, better-organized entries.

This notebook walks through three progressive approaches: first, semantic deduplication to detect and merge near-identical memories; second, pattern abstraction to replace clusters of related specific observations with a single generalized rule; and third, a scheduling mechanism that controls *when* these operations fire in a real agent loop, keeping refactoring costs proportional to its benefit.

In [1]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List, Dict, Optional, Tuple
from datetime import datetime
import os

### Initialize the language model

In [2]:
# Initialize the language model — temperature 0 for consistent, deterministic analyses
llm = ChatOpenAI(model="gpt-4o-mini-2024-07-18", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.0)

## 1. Semantic deduplication
Over the course of an agent's lifetime, the same piece of information is often recorded multiple times - phrased differently each time it comes up. A user preference noted in a Monday session gets re-observed on Thursday, then again the following week. The content is semantically identical, but because the wording differs, a naive equality check would leave all three versions in place. The memory store ends up with three entries doing the work of one, and any retrieval step must sift through all three.

The first level of memory refactoring targets this specific problem. The approach is to compare memory pairs for semantic equivalence using an LLM - not for exact string matching, but for conceptual overlap. Any pair that exceeds a configurable similarity threshold gets consolidated into a single entry that preserves all unique details from both. The merged memory also carries the identifiers of its source entries, so the lineage is never lost.

One important property of this design is its cost profile. Comparing every pair has O(n²) complexity in LLM calls, which becomes expensive for large stores. This is acceptable when refactoring runs on a schedule - for example, nightly or when the store exceeds a size threshold - rather than on every turn. We will revisit how to trigger refactoring efficiently in Section 3.

In [3]:
# --- Data models ---

class MemoryItem(BaseModel):
    """A single stored memory with its provenance metadata."""
    memory_id: str
    content: str
    created_at: datetime
    access_count: int = 0                                        # How many times this memory was retrieved
    source_ids: List[str] = Field(default_factory=list)          # IDs of memories merged into this one


class SimilarityResult(BaseModel):
    """LLM-assessed similarity between two memory items."""
    score: float        # 0.0 (completely different) to 1.0 (identical meaning)
    should_merge: bool  # LLM recommendation — may differ from the numeric threshold
    reasoning: str      # Brief explanation so the decision is auditable


def estimate_tokens(memories: List[MemoryItem]) -> int:
    """Rough token count: ~4 characters per token (consistent with other notebooks in this series)."""
    return sum(len(m.content) for m in memories) // 4


# --- Core functions ---

def assess_similarity(mem1: MemoryItem, mem2: MemoryItem) -> SimilarityResult:
    """Ask the LLM whether two memories express the same idea."""
    prompt = f"""Compare these two memory items for semantic similarity.

Memory A: {mem1.content}
Memory B: {mem2.content}

Score their conceptual overlap from 0.0 (completely different) to 1.0 (same meaning, different words).
Recommend whether merging them into a single entry would preserve all essential information."""

    return llm.with_structured_output(SimilarityResult).invoke(prompt)


def find_duplicate_pairs(
    memories: List[MemoryItem],
    threshold: float = 0.85
) -> List[Tuple[str, str, float]]:
    """
    Identify pairs of memories similar enough to be merged.

    Returns a list of (id1, id2, similarity_score) tuples for all pairs above the threshold.
    The O(n²) comparison is intentional for thoroughness — this function is designed to be called on a schedule, not on every agent turn.
    """
    duplicates = []

    # Iterate over every unique pair (i, j) where i < j to avoid comparing a pair twice
    for i in range(len(memories)):
        for j in range(i + 1, len(memories)):
            result = assess_similarity(memories[i], memories[j])

            # Both the numeric threshold AND the LLM recommendation must agree
            # This dual condition avoids false merges where vocabulary overlaps but meaning differs
            if result.score >= threshold and result.should_merge:
                duplicates.append((memories[i].memory_id, memories[j].memory_id, result.score))

    return duplicates


def merge_pair(mem1: MemoryItem, mem2: MemoryItem) -> MemoryItem:
    """
    Produce a single consolidated memory from two similar ones.
    The merged result carries both source IDs so lineage is always traceable.
    """
    prompt = f"""Merge these two memory entries into one concise, complete memory.
Preserve all unique details from both. Be brief — this will be stored as a single memory item.

Memory A: {mem1.content}
Memory B: {mem2.content}

Return only the merged memory text, nothing else."""

    merged_content = llm.invoke(prompt).content.strip()

    return MemoryItem(
        memory_id=f"merged_{mem1.memory_id}_{mem2.memory_id}",
        content=merged_content,
        created_at=min(mem1.created_at, mem2.created_at),      # Preserve the earlier creation timestamp
        access_count=mem1.access_count + mem2.access_count,    # Carry forward combined retrieval history
        source_ids=[mem1.memory_id, mem2.memory_id]            # Record which memories were consolidated
    )


def deduplicate_memories(
    memories: List[MemoryItem],
    threshold: float = 0.85
) -> Tuple[List[MemoryItem], Dict]:
    """
    Run one full deduplication pass over a list of memories.
    Returns the deduplicated list and a summary of what changed.
    """
    tokens_before = estimate_tokens(memories)

    # Identify all pairs that qualify for merging
    pairs = find_duplicate_pairs(memories, threshold)

    # Build a lookup by ID for fast access during the merge phase
    mem_map = {m.memory_id: m for m in memories}

    consumed: set = set()   # IDs that have already been merged into something else this pass
    result: List[MemoryItem] = []
    merges_performed = 0

    for id1, id2, score in pairs:
        # Skip if either side was already consumed by an earlier merge in this pass
        # Without this guard, a memory could end up merged twice, losing its original content
        if id1 in consumed or id2 in consumed:
            continue

        merged = merge_pair(mem_map[id1], mem_map[id2])
        result.append(merged)

        # Mark both source IDs as consumed so they are not re-used
        consumed.add(id1)
        consumed.add(id2)
        merges_performed += 1

    # Retain every memory that was not consumed by a merge
    for m in memories:
        if m.memory_id not in consumed:
            result.append(m)

    tokens_after = estimate_tokens(result)

    summary = {
        "before": len(memories),
        "after": len(result),
        "pairs_merged": merges_performed,
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "token_reduction": tokens_before - tokens_after,
    }

    return result, summary


# --- Demo ---

initial_memories = [
    MemoryItem(
        memory_id="m1",
        content="The user prefers meetings to be scheduled in the morning, around 9 to 10 AM.",
        created_at=datetime(2024, 1, 15, 9, 0),
        access_count=4
    ),
    MemoryItem(
        memory_id="m2",
        content="User likes having meetings early in the day — ideally between 9:00 and 10:00.",
        created_at=datetime(2024, 1, 17, 10, 30),
        access_count=2
    ),
    MemoryItem(
        memory_id="m3",
        content="User is building a machine learning pipeline in Python using scikit-learn.",
        created_at=datetime(2024, 1, 18, 14, 0),
        access_count=7
    ),
    MemoryItem(
        memory_id="m4",
        content="User mentioned they prefer not to hold meetings after 3 PM.",
        created_at=datetime(2024, 1, 19, 11, 0),
        access_count=1
    ),
]

print("Before deduplication:")
for m in initial_memories:
    print(f"  [{m.memory_id}] {m.content}")

deduped, summary = deduplicate_memories(initial_memories, threshold=0.85)

print(f"\nAfter deduplication:")
for m in deduped:
    src = f"  ← merged from {m.source_ids}" if m.source_ids else ""
    print(f"  [{m.memory_id}] {m.content}{src}")

print(f"\nSummary:")
print(f"  Memories : {summary['before']} → {summary['after']}")
print(f"  Merged   : {summary['pairs_merged']} pair(s)")
print(f"  Tokens   : {summary['tokens_before']} → {summary['tokens_after']}  (saved {summary['token_reduction']} tokens)")

Before deduplication:
  [m1] The user prefers meetings to be scheduled in the morning, around 9 to 10 AM.
  [m2] User likes having meetings early in the day — ideally between 9:00 and 10:00.
  [m3] User is building a machine learning pipeline in Python using scikit-learn.
  [m4] User mentioned they prefer not to hold meetings after 3 PM.

After deduplication:
  [merged_m1_m2] The user prefers meetings to be scheduled early in the day, ideally between 9:00 and 10:00 AM.  ← merged from ['m1', 'm2']
  [m3] User is building a machine learning pipeline in Python using scikit-learn.
  [m4] User mentioned they prefer not to hold meetings after 3 PM.

Summary:
  Memories : 4 → 3
  Merged   : 1 pair(s)
  Tokens   : 71 → 56  (saved 15 tokens)


The deduplication pass operates in three stages. `find_duplicate_pairs` runs an O(n²) comparison: for each unique memory pair, it calls the LLM to score semantic similarity and confirm the merge recommendation. Pairs that exceed the similarity threshold (default 0.85) *and* receive an explicit LLM recommendation are collected. The dual condition - numeric threshold plus LLM confirmation - prevents false merges where two memories happen to share vocabulary but represent different facts.

In the merge stage, `merge_pair` issues a separate LLM call asking for a single consolidated entry. The `consumed` set ensures each original memory participates in at most one merge per pass, preventing a memory from being merged twice within the same run. The merged result carries `source_ids` recording its provenance: if a merged memory later proves incorrect, the originals can be inspected. Access counts are summed so retrieval frequency history is not lost.

## 2. Pattern abstraction
Deduplication addresses the case where the *same* information is stored multiple times. But there is a second, subtler redundancy problem: when an agent records many *related but distinct* observations that collectively point to a single underlying rule. Consider a user who consistently chooses oat milk in coffee drinks. After a dozen sessions, the memory store might contain four separate entries - "ordered oat milk latte Monday," "requested oat milk in espresso Wednesday," "chose oat milk over regular milk Friday," and so on. Each is accurate and unique, but together they express one preference that could be captured in a single sentence: "User always chooses oat milk in coffee."

Pattern abstraction handles this second form of redundancy. Rather than looking for near-identical memories, it looks for clusters of memories that share a common theme and asks whether those memories collectively imply a generalization. When a reliable generalization is found - above a configurable confidence threshold - the supporting specific entries are replaced by the single abstract rule. This does not lose information in any meaningful sense: the specific instances are already captured by the rule, and if an exception ever arises, a new, more precise memory can be added.

The approach uses a two-stage process to keep LLM costs manageable. First, memories are grouped by topic using a single LLM call that assigns cluster labels to all memories at once. Then, within each cluster, a second call decides whether a general rule exists and which specific memories can be safely replaced. This is substantially cheaper than the O(n²) pairwise comparison used in deduplication, making pattern abstraction practical even for moderately large stores.

In [5]:
# --- Additional data models for pattern abstraction ---

class TopicGroup(BaseModel):
    """A cluster of memory IDs that share a common theme."""
    topic_label: str       # Short name for the shared theme (e.g. "coffee preferences")
    memory_ids: List[str]  # IDs of memories belonging to this cluster


class TopicGrouping(BaseModel):
    """Container returned by the grouping call — a list of all clusters."""
    groups: List[TopicGroup]


class PatternCandidate(BaseModel):
    """LLM assessment of whether a topic cluster supports abstraction."""
    has_pattern: bool                                            # True if a reliable generalization was found
    general_rule: Optional[str] = None                          # The abstracted rule, if has_pattern is True
    confidence: float = 0.0                                     # Confidence in the abstraction (0.0–1.0)
    ids_to_replace: List[str] = Field(default_factory=list)     # Memory IDs that can be replaced by this rule


# --- Core functions ---

def group_by_topic(memories: List[MemoryItem]) -> List[TopicGroup]:
    """
    Cluster all memories by shared topic in a single LLM call.

    Using one call for all memories is intentionally cheaper than pairwise comparison —
    topic assignment is a classification task that scales with the number of memories, not n².
    """
    # Format all memories into a numbered list for the prompt
    memory_list = "\n".join(
        f"  ID {m.memory_id}: {m.content}" for m in memories
    )

    prompt = f"""Group the following memories into topic clusters.
Each cluster should contain memories that share the same underlying subject or theme.
Each memory belongs to exactly one cluster.

Memories:
{memory_list}

Return a list of clusters, each with a short topic label and the memory IDs it contains."""

    result = llm.with_structured_output(TopicGrouping).invoke(prompt)
    return result.groups


def extract_pattern(
    group: TopicGroup,
    mem_map: Dict[str, MemoryItem],
    min_confidence: float = 0.75
) -> Optional[PatternCandidate]:
    """
    Decide whether a topic cluster's memories support a general rule worth abstracting.

    Returns a PatternCandidate only if confidence meets the minimum threshold.
    Groups smaller than 3 members are skipped — a pattern built on 2 examples is unreliable.
    """
    # Only consider clusters with enough evidence for a reliable abstraction
    member_ids = [mid for mid in group.memory_ids if mid in mem_map]
    if len(member_ids) < 3:
        return None

    memories_text = "\n".join(
        f"  - {mem_map[mid].content}" for mid in member_ids
    )

    prompt = f"""Analyze these memories from the topic "{group.topic_label}":
{memories_text}

Decide: do these memories collectively imply a single, reliable general rule?
If yes, state the rule concisely and rate your confidence from 0.0 to 1.0.
List which memory IDs can be safely replaced by the general rule.
If the memories are too diverse or evidence is insufficient, set has_pattern to false."""

    candidate = llm.with_structured_output(PatternCandidate).invoke(prompt)

    # Reject low-confidence candidates — a bad abstraction is worse than storing the specifics
    if not candidate.has_pattern or candidate.confidence < min_confidence:
        return None

    return candidate


def abstract_to_patterns(
    memories: List[MemoryItem],
    min_confidence: float = 0.75
) -> Tuple[List[MemoryItem], List[MemoryItem], Dict]:
    """
    Run one full pattern abstraction pass over a list of memories.

    Returns:
        - Updated memory list (specific entries replaced by pattern memories)
        - The new pattern memories (for inspection or logging)
        - A summary of what changed
    """
    tokens_before = estimate_tokens(memories)
    mem_map = {m.memory_id: m for m in memories}

    # Step 1: assign all memories to topic clusters in one LLM call
    groups = group_by_topic(memories)

    replaced_ids: set = set()    # IDs of memories that will be replaced by a pattern
    pattern_memories: List[MemoryItem] = []

    # Step 2: for each cluster, check whether a reliable pattern can be extracted
    for group in groups:
        candidate = extract_pattern(group, mem_map, min_confidence)

        if candidate is None:
            continue  # No reliable pattern found — leave this cluster's memories untouched

        # Create a new MemoryItem that represents the abstracted rule
        pattern_mem = MemoryItem(
            memory_id=f"pattern_{group.topic_label.replace(' ', '_').lower()}",
            content=candidate.general_rule,
            created_at=datetime.now(),
            access_count=0,
            source_ids=[mid for mid in candidate.ids_to_replace if mid in mem_map]  # Track replaced memories
        )
        pattern_memories.append(pattern_mem)
        replaced_ids.update(pattern_mem.source_ids)

    # Keep memories that were not replaced by any pattern
    retained = [m for m in memories if m.memory_id not in replaced_ids]

    # Final list: originals that survived + newly created pattern entries
    result = retained + pattern_memories
    tokens_after = estimate_tokens(result)

    summary = {
        "before": len(memories),
        "after": len(result),
        "patterns_created": len(pattern_memories),
        "memories_replaced": len(replaced_ids),
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "token_reduction": tokens_before - tokens_after,
    }

    return result, pattern_memories, summary


# --- Demo ---

memories_with_patterns = [
    # Four oat-milk observations — expected to abstract into one rule
    MemoryItem(
        memory_id="c1",
        content="User ordered a latte with oat milk on Monday morning.",
        created_at=datetime(2024, 2, 5, 9, 0), access_count=1
    ),
    MemoryItem(
        memory_id="c2",
        content="When offered milk choices, user requested oat milk for their espresso.",
        created_at=datetime(2024, 2, 7, 9, 15), access_count=1
    ),
    MemoryItem(
        memory_id="c3",
        content="User chose oat milk instead of whole milk for their coffee drink.",
        created_at=datetime(2024, 2, 10, 8, 45), access_count=2
    ),
    MemoryItem(
        memory_id="c4",
        content="User again picked oat milk at the café — seems like a consistent preference.",
        created_at=datetime(2024, 2, 12, 9, 30), access_count=1
    ),
    # Unrelated memories that should be retained as-is
    MemoryItem(
        memory_id="u1",
        content="User is building a machine learning pipeline using scikit-learn.",
        created_at=datetime(2024, 2, 8, 14, 0), access_count=5
    ),
    MemoryItem(
        memory_id="u2",
        content="User prefers morning meeting slots, ideally before 10 AM.",
        created_at=datetime(2024, 2, 9, 11, 0), access_count=3
    ),
]

print("Before pattern abstraction:")
for m in memories_with_patterns:
    print(f"  [{m.memory_id}] {m.content}")

result_memories, patterns, summary = abstract_to_patterns(memories_with_patterns, min_confidence=0.75)

print(f"\nAfter pattern abstraction:")
for m in result_memories:
    tag = " [PATTERN]" if m.source_ids else ""
    print(f"  [{m.memory_id}]{tag} {m.content}")
    if m.source_ids:
        print(f"             replaces: {m.source_ids}")

print(f"\nSummary:")
print(f"  Memories : {summary['before']} → {summary['after']}")
print(f"  Patterns created         : {summary['patterns_created']}")
print(f"  Specific memories replaced: {summary['memories_replaced']}")

Before pattern abstraction:
  [c1] User ordered a latte with oat milk on Monday morning.
  [c2] When offered milk choices, user requested oat milk for their espresso.
  [c3] User chose oat milk instead of whole milk for their coffee drink.
  [c4] User again picked oat milk at the café — seems like a consistent preference.
  [u1] User is building a machine learning pipeline using scikit-learn.
  [u2] User prefers morning meeting slots, ideally before 10 AM.

After pattern abstraction:
  [c1] User ordered a latte with oat milk on Monday morning.
  [c2] When offered milk choices, user requested oat milk for their espresso.
  [c3] User chose oat milk instead of whole milk for their coffee drink.
  [c4] User again picked oat milk at the café — seems like a consistent preference.
  [u1] User is building a machine learning pipeline using scikit-learn.
  [u2] User prefers morning meeting slots, ideally before 10 AM.
  [pattern_oat_milk_preference] User consistently prefers oat milk over other 

Pattern abstraction works in two sequential stages. `group_by_topic` handles the first stage: all memories are passed to the LLM in a single call, which assigns each to a topic cluster. This one-shot design is substantially cheaper than pairwise comparison - the cost scales with the number of memories, not with the square of it. The second stage runs `extract_pattern` once per cluster: the LLM reads all members of a cluster together and decides whether they collectively support a reliable generalization. The `min_confidence` threshold (default 0.75) acts as a guard against hasty abstraction - a cluster of loosely related observations should not be collapsed into a rule.

The resulting pattern memory is itself a `MemoryItem`, which means it passes through the same retrieval pipeline as any other memory. Its `source_ids` field records which specific observations it replaced, enabling investigation if the abstraction later proves too coarse. Memories that do not belong to any pattern-eligible cluster are retained unchanged, so the operation is conservative by design.

## 3. Scheduling refactoring in an agent loop
The previous two sections defined the *what* of memory refactoring: how to find and eliminate duplicates, and how to extract general patterns from clusters of related observations. This section addresses the *when* - which is at least as important in practice.

Refactoring is not free. The deduplication step makes O(n²) LLM calls, and pattern abstraction adds another batch on top of that. Running both operations on every agent turn would cost more than the retrieval the agent is actually doing. On the other hand, never refactoring allows memory stores to grow unbounded, degrading every subsequent retrieval. The right answer is to trigger refactoring when the cost is justified by the expected benefit.

Two natural triggers capture most real-world cases. A **memory count threshold** fires when the store grows beyond a certain size - at that point, the retrieval cost of extra entries and the merge savings from refactoring together justify the LLM spend. A **turn interval** fires periodically regardless of store size, catching slow accumulation that would never cross the count threshold on its own. These triggers are complementary: count-based triggering reacts to sudden spikes, while the turn interval handles gradual drift.

The pattern used here - a `RefactoringScheduler` called at the end of each agent turn - mirrors the `CleanupScheduler` introduced in the scheduled context cleanup notebook. The agent loop only calls one method per turn (`maybe_refactor`). If no trigger has fired, that method returns immediately without touching the store or issuing any LLM calls. If a trigger fires, it runs the enabled refactoring stages and resets the counters. The scheduling logic stays entirely separate from the agent's core processing.

In [6]:
# --- Level 3: Refactoring Scheduler ---

class RefactoringConfig(BaseModel):
    """Controls when and what to refactor."""
    max_memories_before_refactor: int = 20    # Fire when store reaches this count
    min_turns_between_refactor: int = 15      # Also fire after this many turns regardless of count
    enable_deduplication: bool = True         # Run semantic deduplication when triggered
    enable_pattern_abstraction: bool = True   # Run pattern abstraction when triggered
    similarity_threshold: float = 0.85        # Passed through to deduplicate_memories
    min_pattern_confidence: float = 0.75      # Passed through to abstract_to_patterns


class MemoryStore:
    """Minimal in-memory store for MemoryItems — just enough to support the scheduler demo."""

    def __init__(self):
        self._memories: List[MemoryItem] = []

    def add(self, memory: MemoryItem) -> None:
        self._memories.append(memory)

    def all(self) -> List[MemoryItem]:
        return list(self._memories)

    def replace_all(self, memories: List[MemoryItem]) -> None:
        """Overwrite the store with the compacted list returned after refactoring."""
        self._memories = memories

    def count(self) -> int:
        return len(self._memories)

    def token_estimate(self) -> int:
        return estimate_tokens(self._memories)


class RefactoringScheduler:
    """
    Decides when to run memory refactoring and executes the enabled stages.

    Call maybe_refactor() at the end of each agent turn.
    The method returns a short status string describing what happened.
    """

    def __init__(self, store: MemoryStore, config: RefactoringConfig):
        self.store = store
        self.config = config
        self._turns_since_last_refactor = 0   # Counts agent turns since the last refactoring run

    def _should_refactor(self) -> Tuple[bool, str]:
        """
        Evaluate both triggers and return (should_run, reason_string).
        Count threshold takes priority — it is checked first.
        """
        if self.store.count() >= self.config.max_memories_before_refactor:
            return True, f"memory count ({self.store.count()}) reached threshold ({self.config.max_memories_before_refactor})"

        if self._turns_since_last_refactor >= self.config.min_turns_between_refactor:
            return True, f"scheduled interval reached ({self._turns_since_last_refactor} turns)"

        # Neither trigger has fired — nothing to do
        return False, ""

    def maybe_refactor(self) -> str:
        """
        Called once at the end of each agent turn.
        Runs refactoring only if a trigger fires; otherwise returns immediately.
        """
        self._turns_since_last_refactor += 1

        should_run, reason = self._should_refactor()

        if not should_run:
            # Common case: neither trigger fired, so no LLM calls are made
            return "no refactoring needed"

        memories = self.store.all()
        count_before = len(memories)
        tokens_before = estimate_tokens(memories)
        stages_run = []

        # Stage 1: deduplicate first so pattern abstraction works on a cleaner input
        if self.config.enable_deduplication:
            memories, dedup_summary = deduplicate_memories(
                memories, threshold=self.config.similarity_threshold
            )
            stages_run.append(f"deduplication (merged {dedup_summary['pairs_merged']} pair(s))")

        # Stage 2: pattern abstraction on whatever survived deduplication
        if self.config.enable_pattern_abstraction:
            memories, patterns, pattern_summary = abstract_to_patterns(
                memories, min_confidence=self.config.min_pattern_confidence
            )
            stages_run.append(f"pattern abstraction ({pattern_summary['patterns_created']} pattern(s))")

        # Write the compacted list back to the store
        self.store.replace_all(memories)

        # Reset the turn counter so the interval trigger measures from this run
        self._turns_since_last_refactor = 0

        count_after = len(memories)
        tokens_after = estimate_tokens(memories)

        return (
            f"REFACTORED — {reason} | "
            f"stages: {', '.join(stages_run)} | "
            f"memories {count_before}→{count_after} | "
            f"tokens {tokens_before}→{tokens_after}"
        )


# --- Demo: simulated agent loop ---

config = RefactoringConfig(
    max_memories_before_refactor=8,   # Low so the demo shows the trigger firing quickly
    min_turns_between_refactor=10,
    enable_deduplication=True,
    enable_pattern_abstraction=True,
)

store = MemoryStore()
scheduler = RefactoringScheduler(store, config)

# Each inner list represents the new memories recorded during that agent turn
new_memories_per_turn = [
    # Turns 1-3: initial distinct observations
    [MemoryItem(memory_id="t1_m1", content="User's primary email is alice@example.com.", created_at=datetime.now(), access_count=0)],
    [MemoryItem(memory_id="t2_m1", content="User is working on a Python data pipeline project.", created_at=datetime.now(), access_count=0)],
    [MemoryItem(memory_id="t3_m1", content="User ordered coffee with oat milk this morning.", created_at=datetime.now(), access_count=0)],
    # Turns 4-6: more observations, some overlapping with earlier ones
    [MemoryItem(memory_id="t4_m1", content="User prefers oat milk in all coffee drinks.", created_at=datetime.now(), access_count=0)],
    [MemoryItem(memory_id="t5_m1", content="User is building a data pipeline using Apache Spark and Python.", created_at=datetime.now(), access_count=0)],
    [MemoryItem(memory_id="t6_m1", content="User mentioned they like morning meeting slots before 10 AM.", created_at=datetime.now(), access_count=0)],
    # Turns 7-9: accumulation continues, nearing the count threshold
    [MemoryItem(memory_id="t7_m1", content="User again chose oat milk for their latte.", created_at=datetime.now(), access_count=0)],
    [MemoryItem(memory_id="t8_m1", content="User's email address is alice@example.com — confirmed again.", created_at=datetime.now(), access_count=0)],
    [MemoryItem(memory_id="t9_m1", content="User prefers meetings in the morning, before 10.", created_at=datetime.now(), access_count=0)],
    # Turns 10-12: continued accumulation after refactoring resets the counter
    [MemoryItem(memory_id="t10_m1", content="User requested oat milk instead of regular milk in their cappuccino.", created_at=datetime.now(), access_count=0)],
    [MemoryItem(memory_id="t11_m1", content="User mentioned using scikit-learn for model training in their project.", created_at=datetime.now(), access_count=0)],
    [MemoryItem(memory_id="t12_m1", content="User prefers morning hours for any scheduled calls.", created_at=datetime.now(), access_count=0)],
]

print(f"Config: refactor when store reaches {config.max_memories_before_refactor} memories "
      f"or every {config.min_turns_between_refactor} turns\n")

for turn_num, new_mems in enumerate(new_memories_per_turn, start=1):
    # Agent adds observations from this turn to the store
    for m in new_mems:
        store.add(m)

    # End of turn: let the scheduler decide whether to refactor
    status = scheduler.maybe_refactor()

    print(f"Turn {turn_num:2d} | store: {store.count():2d} memories, ~{store.token_estimate()} tokens | {status}")

Config: refactor when store reaches 8 memories or every 10 turns

Turn  1 | store:  1 memories, ~10 tokens | no refactoring needed
Turn  2 | store:  2 memories, ~23 tokens | no refactoring needed
Turn  3 | store:  3 memories, ~34 tokens | no refactoring needed
Turn  4 | store:  4 memories, ~45 tokens | no refactoring needed
Turn  5 | store:  5 memories, ~61 tokens | no refactoring needed
Turn  6 | store:  6 memories, ~76 tokens | no refactoring needed
Turn  7 | store:  7 memories, ~86 tokens | no refactoring needed
Turn  8 | store:  8 memories, ~101 tokens | REFACTORED — memory count (8) reached threshold (8) | stages: deduplication (merged 1 pair(s)), pattern abstraction (1 pattern(s)) | memories 8→8 | tokens 101→101
Turn  9 | store:  8 memories, ~98 tokens | REFACTORED — memory count (9) reached threshold (8) | stages: deduplication (merged 2 pair(s)), pattern abstraction (1 pattern(s)) | memories 9→8 | tokens 113→98
Turn 10 | store:  9 memories, ~115 tokens | REFACTORED — memory cou

The `RefactoringScheduler` separates two concerns that would otherwise be tangled: the decision of *when* to refactor and the mechanics of *how* to do it. The store and config are injected at construction time, and the only method the agent loop calls is `maybe_refactor` - one line at the end of each turn. Internally, `_should_refactor` evaluates the count threshold first (priority trigger) and the turn interval second (fallback). If neither fires, the method returns in microseconds without touching the store or issuing any LLM calls.

When a trigger fires, the stages run in sequence: deduplication first, pattern abstraction second. This order is intentional - deduplication removes near-duplicate entries before grouping runs, so the topic clustering step operates on a smaller, cleaner input. After both stages complete, the compacted list is written back to the store via `replace_all`, and the turn counter resets to zero so the interval trigger measures from this run forward.

### Key takeaways
- **What each stage does and when to apply it**: Semantic deduplication targets near-identical entries - the same fact recorded multiple times with different phrasing. It is thorough but has O(n²) LLM call cost, so it should run on a schedule or when the store crosses a count threshold, not on every turn. Pattern abstraction targets clusters of related but distinct observations that collectively imply a general rule. It uses a two-stage approach (one grouping call, then one call per cluster) and is cheaper per memory item than deduplication. Both stages are most effective when run together: deduplication first cleans up duplicate phrasing, then pattern abstraction generalizes what remains.
- **Cost and sizing**: At 20 memories, deduplication can require up to 190 pairwise LLM comparisons. The `max_memories_before_refactor` threshold in the scheduler should be set with this in mind - larger thresholds reduce the frequency of refactoring runs but increase the cost of each one. Pattern abstraction is substantially cheaper: its call count grows with the number of topic clusters, which is typically far smaller than n. For most agents, a threshold between 15 and 30 memories strikes a reasonable balance between store quality and LLM spend.
- **Combining refactoring with other refresh strategies**: Memory refactoring and scheduled context cleanup are complementary, not competing. Cleanup decides *which* memories to keep based on age or importance scores; refactoring reorganizes *how* the surviving memories are stored. A well-designed memory system uses cleanup to prune and refactoring to consolidate, maintaining both freshness and compactness over time.